# einops-reduce — ex5: row-wise softmax stabilization (reduce + keepdim + broadcast)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.reduce` patterns that ramp from single-axis mean → multi-axis global pool → keepdim broadcast → decomposed average pool → softmax stabilization. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-reduce`**, which bridges to the bank subtopic `Einops: Reduce` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, reduction, **axes_lengths)` collapses named axes:
1. **Single-axis drop** — `'h w c -> h w'` with `'mean'` averages channels.
2. **Multi-axis drop** — `'b c h w -> b'` reduces three axes at once.
3. **Keepdim placeholder** — `'b c h w -> b c () ()'` keeps size-1 axes for broadcasting.
4. **Decompose-then-reduce** — `'b c (h h2) (w w2) -> b c h w'` with `h2=2, w2=2` does 2×2 pooling.

Reduction strings: `'mean'`, `'sum'`, `'max'`, `'min'`, `'prod'`, `'any'`, `'all'`, or a callable.
Any axis that appears on the left but not on the right is reduced over.

### Exercise 5 — row-wise softmax stabilization (reduce + keepdim + broadcast)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize reduce + keepdim + broadcast subtraction to perform numerically-stable softmax preprocessing.
> Keywords: softmax-stabilize, broadcast-subtract, integration, multi-kc
> ```

**KCs targeted:** `reduce-pick-aggregator`, `reduce-keepdim-broadcast`, `reduce-normalize-pattern`

Implement `ex5_softmax_stabilize(x)` to subtract the per-row maximum from every element. This is the standard pre-softmax stabilization step that keeps `exp(x)` from overflowing.

Input shape: `(b, n)`. Output shape: `(b, n)`. After your transform, every row's maximum should be exactly `0`.

Use `einops.reduce` with `()` on the row axis so the per-row max keeps a size-1 placeholder, then broadcast-subtract.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one expression; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_softmax_stabilize(x: Tensor) -> Tensor:
    row_max = reduce(x, 'b n -> b ()', 'max')
    return x - row_max


<details><summary>Solution</summary>

```python
def ex5_softmax_stabilize(x: Tensor) -> Tensor:
    row_max = reduce(x, 'b n -> b ()', 'max')
    return x - row_max
```

**Reading the pattern.**
- `'b n -> b ()'` reduces over `n` and keeps a size-1 placeholder, giving `row_max` shape `(b, 1)`.
- `x - row_max` then broadcasts the per-row max across all columns.

**Why this matters.** `exp(x_i)` for large positive `x_i` overflows. Subtracting the per-row max is mathematically a no-op for softmax (numerator and denominator both scale by `exp(-row_max)`) but keeps every exponent ≤ 0, so `exp` stays in `(0, 1]`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()